# C2 - StudentLife ROBUSTNESS CHECK (step 5)

**Question:** in step 4 the Random Forest found a weak signal (AUC about 0.63). Is that signal **real behaviour**, or just an artifact (for example the amount of data the phone collected)? And does your **personal-baseline idea** (`dev_` features) actually help?

## What this notebook does
1. Builds 4 feature sets and compares them:
   - **A** all 18 features (same as step 4)
   - **B** without "coverage" features. `act_n`, `app_n_snapshots`, `app_n_distinct`, `dev_app_n_snapshots` mostly say HOW MUCH data was collected, not what the person did.
   - **C** raw features only (no coverage, no `dev_`)
   - **D** personal-baseline `dev_` features only (no coverage)
2. Uses **5 different random participant-grouped fold layouts x 5 folds = 25 test folds** per model, to see how stable the numbers are.
3. For each, also runs a **shuffled-label control** (labels randomly mixed). This is the real "chance level" here.

## Rule decided BEFORE looking at results
A result counts as **clearly above chance** only if `mean AUC > shuffled mean + 2 x shuffled std` (both over the 25 folds). Otherwise we say "no reliable signal".

Everything is PRELIMINARY. Target = stress label from StudentLife (not C2 capacity). Participant ID is never a feature.

## Cell 1 - Load the step 3 table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT_DIR = Path('/content/drive/MyDrive/AnchorAI_c2_outputs')
table = pd.read_csv(OUT_DIR / 'studentlife_model_table.csv')
with open(OUT_DIR / 'studentlife_split.json') as fh:
    FEATURES = json.load(fh)['features']

y, groups = table['y_stressed'], table['uid']
print('rows:', len(table), '| participants:', groups.nunique(), '| all features:', len(FEATURES))

per_person = table.groupby('uid')['y_stressed'].agg(['mean', 'size'])
print('\nShare of "stressed" answers PER PERSON (people differ a lot):')
print(per_person['mean'].describe().round(2))
print('people with >=80% stressed:', int((per_person['mean'] >= 0.8).sum()),
      '| people with <=20% stressed:', int((per_person['mean'] <= 0.2).sum()))

## Cell 2 - Define the feature sets

In [ ]:
COVERAGE = ['act_n', 'app_n_snapshots', 'app_n_distinct', 'dev_app_n_snapshots']
SETS = {
    'A all features':               FEATURES,
    'B no coverage':                [f for f in FEATURES if f not in COVERAGE],
    'C raw only (no cov, no dev_)': [f for f in FEATURES if f not in COVERAGE and not f.startswith('dev_')],
    'D dev_ only (no coverage)':    [f for f in FEATURES if f.startswith('dev_') and f not in COVERAGE],
}
for k, v in SETS.items():
    print(f'{k}: {len(v)} features')
    print('   ', v)
assert all('uid' not in v for v in SETS.values())

## Cell 3 - Models and repeated grouped cross-validation
Same two models as step 4 (preparation steps are inside the pipeline, so nothing leaks). Random Forest uses 200 trees here to keep it fast.
**This cell takes a few minutes** (it fits many models).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score

def make_model(name):
    if name == 'Logistic Regression':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                         ('clf', LogisticRegression(max_iter=1000, random_state=42))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('clf', RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1))])

scoring = {'roc_auc': 'roc_auc', 'f1': make_scorer(f1_score, zero_division=0)}
SEEDS = [0, 1, 2, 3, 4]

def repeated_cv(model_name, cols, labels):
    aucs, f1s = [], []
    for s in SEEDS:
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
        res = cross_validate(make_model(model_name), table[cols], labels, groups=groups, cv=cv, scoring=scoring)
        aucs += list(res['test_roc_auc']); f1s += list(res['test_f1'])
    return np.array(aucs), np.array(f1s)

rows = []
for set_name, cols in SETS.items():
    for model_name in ('Logistic Regression', 'Random Forest'):
        auc, f1 = repeated_cv(model_name, cols, y)
        rows.append({'feature set': set_name, 'model': model_name, 'n_features': len(cols),
                     'AUC mean': auc.mean(), 'AUC std': auc.std(), 'F1 mean': f1.mean(), 'F1 std': f1.std()})
    print('done:', set_name)

results = pd.DataFrame(rows)
display(results.round(3))

## Cell 4 - Shuffled-label control (what does "no signal" look like here?)
Same feature sets and folds, but the labels are shuffled across answers. If the real result is not clearly higher than this, there is no reliable signal.

In [ ]:
ctrl_rows = []
for set_name, cols in SETS.items():
    for model_name in ('Logistic Regression', 'Random Forest'):
        all_auc = []
        for s in SEEDS:
            y_shuf = pd.Series(np.random.RandomState(100 + s).permutation(y.values), index=y.index)
            cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
            res = cross_validate(make_model(model_name), table[cols], y_shuf, groups=groups, cv=cv, scoring={'roc_auc': 'roc_auc'})
            all_auc += list(res['test_roc_auc'])
        all_auc = np.array(all_auc)
        ctrl_rows.append({'feature set': set_name, 'model': model_name,
                          'shuffled AUC mean': all_auc.mean(), 'shuffled AUC std': all_auc.std()})
ctrl = pd.DataFrame(ctrl_rows)
final = results.merge(ctrl, on=['feature set', 'model'])
final['threshold (mean+2std)'] = final['shuffled AUC mean'] + 2 * final['shuffled AUC std']
final['clearly above chance?'] = np.where(final['AUC mean'] > final['threshold (mean+2std)'], 'YES', 'no')
display(final[['feature set', 'model', 'AUC mean', 'AUC std', 'F1 mean', 'shuffled AUC mean', 'threshold (mean+2std)', 'clearly above chance?']].round(3))
final.round(4).to_csv(OUT_DIR / 'studentlife_robustness_results.csv', index=False)

## Cell 5 - Picture: real AUC vs shuffled control
Bars = real mean AUC (lines = spread over the 25 folds). Dashed black line = 0.5 (guessing). Grey markers = shuffled control.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4.5))
labels = [f"{r['feature set']}\n{r['model'].replace('Logistic Regression','LR').replace('Random Forest','RF')}" for _, r in final.iterrows()]
x = np.arange(len(final))
ax.bar(x, final['AUC mean'], yerr=final['AUC std'], capsize=3, color='tab:blue', alpha=0.8, label='real labels')
ax.scatter(x, final['shuffled AUC mean'], color='grey', zorder=3, label='shuffled labels (control)')
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=8)
ax.set_ylabel('ROC-AUC (grouped CV)'); ax.set_ylim(0.4, 0.75); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'studentlife_robustness_auc.png', dpi=120)
plt.show()

## Cell 6 - What to send back to Claude
Send the printed output of Cells 1, 4 and the picture of Cell 5.